# Unified Base Hard Negative Pools

목표:

- 이미 생성한 `fixed_data/c3_pools`와 `fixed_data/base_hard_pools`를 재활용한다.
- Case1 / Case3를 학습용 negative pool에서는 구분하지 않고 **src 기준 하나의 base pool**로 합친다.
- real positive는 unified pool에서 제거한다.
- Case2 candidate 정보는 별도로 보존한다.
- 이후 Case2 pseudo가 `(new_src, known_dst)`로 채택되었는데 `new_src`가 unified pool에 없으면,
  **그 src 하나만 lazy-build하여 해당 fold pool에 편입**한다.
- iterative round마다 전체 pool을 rebuild하지 않는다.

주의:
이 notebook은 현재 생성된 Case1 base pool의 모집단이 최종 확정되었다는 전제에서 실행한다.

In [1]:
from pathlib import Path
import sys, pickle, json
import pandas as pd

ROOT = Path("/home2/jmbyeon/developments/workspace/apps/uspto_litigation/소송예측모델 개발/소송예측9월/0918PSEUDO_ITERATIVE")
assert ROOT.exists(), ROOT
sys.path.insert(0, str(ROOT))

import config
from data import load_snapshot

C3_DIR = ROOT / "fixed_data" / "c3_pools"
CASE12_DIR = ROOT / "fixed_data" / "base_hard_pools"
OUT_DIR = ROOT / "fixed_data" / "unified_base_pools"
CASE2_DIR = ROOT / "fixed_data" / "case2_candidates"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CASE2_DIR.mkdir(parents=True, exist_ok=True)

FOLDS = config.TRAIN_FOLDS

print("C3_DIR    :", C3_DIR)
print("CASE12_DIR:", CASE12_DIR)
print("OUT_DIR   :", OUT_DIR)
print("CASE2_DIR :", CASE2_DIR)
print("FOLDS     :", FOLDS)

C3_DIR    : /home2/jmbyeon/developments/workspace/apps/uspto_litigation/소송예측모델 개발/소송예측9월/0918PSEUDO_ITERATIVE/fixed_data/c3_pools
CASE12_DIR: /home2/jmbyeon/developments/workspace/apps/uspto_litigation/소송예측모델 개발/소송예측9월/0918PSEUDO_ITERATIVE/fixed_data/base_hard_pools
OUT_DIR   : /home2/jmbyeon/developments/workspace/apps/uspto_litigation/소송예측모델 개발/소송예측9월/0918PSEUDO_ITERATIVE/fixed_data/unified_base_pools
CASE2_DIR : /home2/jmbyeon/developments/workspace/apps/uspto_litigation/소송예측모델 개발/소송예측9월/0918PSEUDO_ITERATIVE/fixed_data/case2_candidates
FOLDS     : [(2004, 2005, 2006), (2006, 2007, 2008), (2008, 2009, 2010), (2010, 2011, 2012)]


## 1. 기존 pool 파일 존재 확인

In [2]:
for cutoff, ys, ye in FOLDS:
    c3 = C3_DIR / f"cutoff_{cutoff}" / "c3_data.pkl"
    c12 = CASE12_DIR / f"pool_{cutoff}.pkl"
    print(cutoff, "C3:", c3.exists(), "CASE12:", c12.exists())
    assert c3.exists(), c3
    assert c12.exists(), c12
print("✓ all input files exist")

2004 C3: True CASE12: True
2006 C3: True CASE12: True
2008 C3: True CASE12: True
2010 C3: True CASE12: True
✓ all input files exist


## 2. C3 src pool 추출

`train_records`와 `val_records`에 들어 있는 `final_train_hard`를 src 기준 union한다.

원본 C3 파일은 baseline 재현을 위해 수정하지 않는다.

In [3]:
def load_c3_src_pools(cutoff):
    path = C3_DIR / f"cutoff_{cutoff}" / "c3_data.pkl"
    with open(path, "rb") as f:
        obj = pickle.load(f)

    pools = {}

    # train + val의 src 후보를 모두 base source universe에 포함.
    # validation에서 실제 사용할 hard/random negative 자체는 fixed val_sets가 계속 담당한다.
    for rec in obj.get("train_records", []) + obj.get("val_records", []):
        s = int(rec["src"])
        hp = {int(x) for x in rec.get("final_train_hard", [])}
        pools.setdefault(s, set()).update(hp)

    real_pairs = {(int(s), int(d)) for s, d in obj["all_positive_pairs"]}

    real_by_src = {}
    for s, d in real_pairs:
        real_by_src.setdefault(s, set()).add(d)

    return pools, real_pairs, real_by_src

## 3. Case1 pool + Case2 records 읽기

앞서 만든 `base_hard_pools/pool_XXXX.pkl`을 그대로 사용한다.

In [4]:
def load_case12_base(cutoff):
    path = CASE12_DIR / f"pool_{cutoff}.pkl"
    with open(path, "rb") as f:
        return pickle.load(f)

## 4. src 기준 Unified Pool 생성

Case1/C3 출처는 metadata 통계로만 남기고, 실제 학습 pool은 `pools[src]` 하나만 사용한다.

최종 base pool에서도 real positive와 self-loop는 제거한다.

In [5]:
def build_unified_fold(cutoff, ys, ye):
    c3_pools, real_pairs, real_by_src = load_c3_src_pools(cutoff)
    c12 = load_case12_base(cutoff)

    case1_pools = c12.get("case1_src_pools", {})
    case2_records = c12.get("case2_records", [])

    all_src = set(c3_pools) | {int(s) for s in case1_pools.keys()}

    unified = {}
    source_meta = {}

    for s in sorted(all_src):
        c3 = set(c3_pools.get(s, set()))

        c1_obj = case1_pools.get(s)
        if c1_obj is None:
            c1_obj = case1_pools.get(str(s), {})
        c1 = {int(x) for x in c1_obj.get("hard_pool", [])}

        base = c3 | c1

        # safety: src 자신 + 모든 known real positive 제거
        base.discard(s)
        base -= real_by_src.get(s, set())

        unified[s] = sorted(base)

        source_meta[s] = {
            "from_c3": s in c3_pools,
            "from_case1": bool(c1_obj),
            "n_c3_candidates": len(c3),
            "n_case1_candidates": len(c1),
            "n_unified": len(base),
        }

    payload = {
        "cutoff": cutoff,
        "target": [ys, ye],
        "pools": unified,
        "real_positive_by_src": {
            int(s): sorted(ds) for s, ds in real_by_src.items()
        },
        "source_meta": source_meta,
        "num_patents": int(c12.get("num_patents", 0)),
        "policy": {
            "key": "src",
            "case1_case3_unified": True,
            "real_positive_removed": True,
            "case2_new_src_policy": "lazy_build_and_add",
            "round_rebuild": False,
        },
    }

    out = OUT_DIR / f"pool_{cutoff}.pkl"
    with open(out, "wb") as f:
        pickle.dump(payload, f)

    # Case2는 src 미정 상태이므로 candidate 정보만 별도 보존
    c2out = CASE2_DIR / f"cases_{cutoff}.pkl"
    with open(c2out, "wb") as f:
        pickle.dump({
            "cutoff": cutoff,
            "target": [ys, ye],
            "records": case2_records,
        }, f)

    overlap = set(c3_pools) & {int(s) for s in case1_pools.keys()}

    summary = {
        "cutoff": cutoff,
        "target": f"{ys}-{ye}",
        "c3_unique_src": len(c3_pools),
        "case1_unique_src": len(case1_pools),
        "case1_c3_overlap_src": len(overlap),
        "unified_unique_src": len(unified),
        "case2_records": len(case2_records),
        "real_positive_pairs": len(real_pairs),
        "empty_unified_pools": sum(len(v) == 0 for v in unified.values()),
    }

    (OUT_DIR / f"pool_{cutoff}_summary.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )

    print(json.dumps(summary, indent=2))
    return summary

## 5. 4개 fold unified pool 생성

In [6]:
summaries = []
for cutoff, ys, ye in FOLDS:
    print("\n" + "="*90)
    print(f"UNIFY {cutoff} -> {ys}-{ye}")
    print("="*90)
    summaries.append(build_unified_fold(cutoff, ys, ye))

summary_df = pd.DataFrame(summaries)
display(summary_df)
summary_df.to_csv(OUT_DIR / "unified_pool_summary.csv", index=False)


UNIFY 2004 -> 2005-2006
{
  "cutoff": 2004,
  "target": "2005-2006",
  "c3_unique_src": 49,
  "case1_unique_src": 1746,
  "case1_c3_overlap_src": 6,
  "unified_unique_src": 1789,
  "case2_records": 389,
  "real_positive_pairs": 187,
  "empty_unified_pools": 32
}

UNIFY 2006 -> 2007-2008
{
  "cutoff": 2006,
  "target": "2007-2008",
  "c3_unique_src": 46,
  "case1_unique_src": 2095,
  "case1_c3_overlap_src": 14,
  "unified_unique_src": 2127,
  "case2_records": 609,
  "real_positive_pairs": 295,
  "empty_unified_pools": 33
}

UNIFY 2008 -> 2009-2010
{
  "cutoff": 2008,
  "target": "2009-2010",
  "c3_unique_src": 37,
  "case1_unique_src": 2272,
  "case1_c3_overlap_src": 6,
  "unified_unique_src": 2303,
  "case2_records": 552,
  "real_positive_pairs": 354,
  "empty_unified_pools": 32
}

UNIFY 2010 -> 2011-2012
{
  "cutoff": 2010,
  "target": "2011-2012",
  "c3_unique_src": 75,
  "case1_unique_src": 2958,
  "case1_c3_overlap_src": 18,
  "unified_unique_src": 3015,
  "case2_records": 743,
  

,cutoff,target,c3_unique_src,case1_unique_src,case1_c3_overlap_src,unified_unique_src,case2_records,real_positive_pairs,empty_unified_pools
0,2004,2005-2006,49,1746,6,1789,389,187,32
1,2006,2007-2008,46,2095,14,2127,609,295,33
2,2008,2009-2010,37,2272,6,2303,552,354,32
3,2010,2011-2012,75,2958,18,3015,743,656,37


## 6. Unified pool safety verification

반드시:

`unified negative ∩ real positive = 0`

이어야 한다.

In [7]:
checks = []

for cutoff, ys, ye in FOLDS:
    with open(OUT_DIR / f"pool_{cutoff}.pkl", "rb") as f:
        obj = pickle.load(f)

    collisions = set()
    self_loops = set()

    for s, pool in obj["pools"].items():
        s = int(s)
        pool = {int(x) for x in pool}
        real = set(obj["real_positive_by_src"].get(s, []))

        for d in pool & real:
            collisions.add((s, d))

        if s in pool:
            self_loops.add((s, s))

    checks.append({
        "fold": f"{cutoff}->{ys}-{ye}",
        "src": len(obj["pools"]),
        "real_positive_collision": len(collisions),
        "self_loop_collision": len(self_loops),
    })

check_df = pd.DataFrame(checks)
display(check_df)

assert check_df["real_positive_collision"].sum() == 0
assert check_df["self_loop_collision"].sum() == 0

print("✓ unified pool verification passed")

,fold,src,real_positive_collision,self_loop_collision
0,2004->2005-2006,1789,0,0
1,2006->2007-2008,2127,0,0
2,2008->2009-2010,2303,0,0
3,2010->2011-2012,3015,0,0


✓ unified pool verification passed


# 7. Case2 신규 src용 lazy-build helper

아래 함수들은 다음 iterative 코드에서도 그대로 `.py`로 옮겨 쓸 수 있다.

Case2 pseudo가 `(src, known_dst)`로 채택되었을 때:

1. `src`가 unified pool에 있으면 그대로 사용
2. 없으면 해당 cutoff snapshot에서 **src 하나의 A/B/C/D pool만 생성**
3. real positives 제거
4. `pool_XXXX.pkl`의 `pools[src]`에 추가
5. 이후 round에서는 재사용

주의: Case2 pseudo positive 자체는 base pool을 영구 삭제해서 변형하기보다,
학습 시 `base_pool - cumulative_positive_by_src[src]`로 제외한다.

In [ ]:
def norm(x):
    if pd.isna(x): return None
    s = str(x)
    return s[:-2] if s.endswith(".0") else s

def relation_maps(obj):
    d=obj["data"]; p2ipc={}; ipc2p={}; adj={}
    if ("patent","has_ipc","ipc") in d.edge_types:
        for p,i in d["patent","has_ipc","ipc"].edge_index.cpu().t().tolist():
            p2ipc.setdefault(int(p),set()).add(int(i))
            ipc2p.setdefault(int(i),set()).add(int(p))
    for et in [("patent","cites","patent"),("patent","cited_by","patent")]:
        if et in d.edge_types:
            for s,t in d[et].edge_index.cpu().t().tolist():
                adj.setdefault(int(s),set()).add(int(t))
    return p2ipc,ipc2p,adj

def khop(adj, src, k=3):
    seen={int(src)}; frontier={int(src)}; layers=[]
    for _ in range(k):
        nxt=set()
        for x in frontier:
            nxt |= adj.get(x,set())
        nxt -= seen
        layers.append(nxt); seen |= nxt; frontier=nxt
        if not frontier:
            layers += [set()]*(k-len(layers)); break
    return layers

def ownership_groups():
    oc=pd.read_parquet(config.OWNERSHIP_PATH).copy()
    ren={}
    if "patent" in oc.columns and "patent_id" not in oc.columns:
        ren["patent"]="patent_id"
    if "assignee" in oc.columns and "assignee_id" not in oc.columns:
        ren["assignee"]="assignee_id"
    oc=oc.rename(columns=ren)
    oc["patent_id"]=oc["patent_id"].map(norm)
    oc["assignee_id"]=oc["assignee_id"].map(norm)
    oc["start_dt"]=pd.to_datetime(oc["start_dt"],errors="coerce")
    oc["end_dt"]=pd.to_datetime(oc["end_dt"],errors="coerce")
    return {str(a):g for a,g in oc.groupby("assignee_id",sort=False)}

def portfolio(groups, aid, cutoff_dt, p2i):
    aid=norm(aid)
    if aid is None or aid not in groups:
        return set()
    q=groups[aid]
    q=q[
        (q.start_dt.isna()|q.start_dt.le(cutoff_dt)) &
        (q.end_dt.isna()|q.end_dt.gt(cutoff_dt))
    ]
    return {
        p2i[str(p)]
        for p in q.patent_id.dropna().astype(str).unique()
        if str(p) in p2i
    }

In [ ]:
def build_single_src_pool(
    src,
    candidate_assignee_id,
    base_snapshot,
    ownership,
    cutoff,
    real_positive_by_src,
):
    """
    Case2에서 새 pseudo src가 등장했을 때 src 하나만 lazy-build.
    기존 C3/Case1 A/B/C/D 규칙과 동일.
    """
    src = int(src)

    p2i = {str(k): int(v) for k, v in base_snapshot["pat2idx"].items()}
    p2ipc, ipc2p, adj = relation_maps(base_snapshot)
    cutoff_dt = pd.Timestamp(f"{cutoff}-12-30")

    same=set()
    for ip in p2ipc.get(src,set()):
        same |= ipc2p.get(ip,set())

    layers=khop(adj,src,3)
    c1=layers[0] if len(layers)>0 else set()
    c2=c1|(layers[1] if len(layers)>1 else set())
    c3=c2|(layers[2] if len(layers)>2 else set())

    pf=portfolio(
        ownership,
        candidate_assignee_id,
        cutoff_dt,
        p2i,
    )

    real={int(x) for x in real_positive_by_src.get(src, [])}
    forbid={src}|real

    A=(same&c1)-forbid
    B=(same&c2)-forbid-A
    C=(same&c3)-forbid-A-B
    D=(pf&(same|c3))-forbid-A-B-C

    return sorted(A|B|C|D)

In [ ]:
def ensure_case2_src_pool(cutoff, src, candidate_assignee_id):
    """
    unified pool에 src가 없을 때만 생성하여 해당 fold 파일에 편입.
    """
    path = OUT_DIR / f"pool_{cutoff}.pkl"

    with open(path, "rb") as f:
        obj = pickle.load(f)

    src = int(src)

    if src in obj["pools"]:
        return {
            "created": False,
            "src": src,
            "n_hard": len(obj["pools"][src]),
        }

    base = load_snapshot(config.SNAP_DIR, cutoff)
    og = ownership_groups()

    new_pool = build_single_src_pool(
        src=src,
        candidate_assignee_id=candidate_assignee_id,
        base_snapshot=base,
        ownership=og,
        cutoff=cutoff,
        real_positive_by_src=obj["real_positive_by_src"],
    )

    obj["pools"][src] = new_pool
    obj["source_meta"][src] = {
        "from_c3": False,
        "from_case1": False,
        "from_case2_lazy": True,
        "n_unified": len(new_pool),
    }

    with open(path, "wb") as f:
        pickle.dump(obj, f)

    return {
        "created": True,
        "src": src,
        "n_hard": len(new_pool),
    }

print("✓ ensure_case2_src_pool() ready")

## 8. Iterative round에서 실제 negative를 가져오는 방식

Base pool은 그대로 두고, **누적 positive만 매 round 빼준다.**

In [ ]:
def get_available_negatives(unified_obj, src, cumulative_pseudo_by_src):
    src = int(src)

    base = set(unified_obj["pools"].get(src, []))
    real = set(unified_obj["real_positive_by_src"].get(src, []))
    pseudo = set(cumulative_pseudo_by_src.get(src, []))

    neg = base - real - pseudo

    # safety
    assert not (neg & real)
    assert not (neg & pseudo)
    assert src not in neg

    return sorted(neg)

print("✓ available negative = BASE_POOL[src] - ALL_POSITIVE[src]")

## 완료 후 확인할 것

마지막으로 다음 파일이 생성되어야 한다.

```text
fixed_data/
├── unified_base_pools/
│   ├── pool_2004.pkl
│   ├── pool_2006.pkl
│   ├── pool_2008.pkl
│   ├── pool_2010.pkl
│   └── unified_pool_summary.csv
│
└── case2_candidates/
    ├── cases_2004.pkl
    ├── cases_2006.pkl
    ├── cases_2008.pkl
    └── cases_2010.pkl
```

다음 단계는 **Round 0 → Case1/2 inference → pseudo 선택 → cumulative positive 갱신 → student 재학습**을 담당하는 iterative runner를 만드는 것이다.